# Split maps of CYGNSS data: two days over South Sumatra and two months over the Mekong

Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import geemap
import ee
import cmocean
import matplotlib as mpl
import numpy as np

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")

Set up the functions to (i) remove NaN values from CYGNSS data, (ii) convert inundation data to fractional inundation, and (iii) remove bias from the fractional inundation data (in case of cygnss anomalies).

In [4]:
def mask_inundation(image):
    return (
        image.updateMask(image.neq(255))
             .divide(100)
             .copyProperties(image, ['system:time_start'])
    )

def mask_anomalies(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    image = image.subtract(100)               # remove bias
    return image.divide(100)                  # scale to fractional values


#### Split map of CYGNSS data: two days over South Sumatra

Select the dates of interest for the two images, filter the collection accordingly, and select one of the following bands: inundation_1day, inundation_3day, interpolation_flag, inundation_anomaly_1day or inundation_anomaly_3day.

In [5]:
#dates
date1 = '2025-11-13'
date2 = '2025-11-14'

#images from collection
image1 = cygnss_collection.filterDate(date1, ee.Date(date1).advance(1, 'day')).first() # the syntax of the function is Collection.filterDate(start, stop).first() 
image2 = cygnss_collection.filterDate(date2, ee.Date(date2).advance(1, 'day')).first()

#bands from images
anomalies_day1 = mask_anomalies(image1.select('inundation_anomaly_1day')) # selecting the appropriate band and applying a function to mask no-data values, removing bias, and scale to fractional inundation
anomalies_day2 = mask_anomalies(image2.select('inundation_anomaly_1day')) #as above

Set the visualization parameters (colors and value ranges) for the plotted bands. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. The code below converts Matplotlib colormaps to HEX color codes. Visualization parameters are defined for inundation anomaly data (nominal range: −1 to 1).

In [6]:
anomaly_palette = cmocean.cm.balance_r(np.linspace(0, 1, 256)) #palette (balance) from matplotlib (https://matplotlib.org/cmocean/)
anomaly_palette_hex = [mpl.colors.rgb2hex(color) for color in anomaly_palette]

anomaly_vis = {
    'min': -0.2,  
    'max': 0.2,  
    'palette': anomaly_palette_hex  # Paleta kolorów dla anomalii (niebieski-czerwony)
}

Plot split map with CYGNSS inundation anomalies from two different days. Please note that when plotting observational data (which contains a large number of no-data pixels), it may be necessary to zoom in to properly visualize the colors.

In [7]:
# Region of interest – South Sumatra
roi = ee.Geometry.Rectangle([
    105, -6,   # west, south
    106, -5     # east, north
])

Map = geemap.Map() # generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
left_layer = geemap.ee_tile_layer(anomalies_day1, anomaly_vis, "DOF inundation anomalies") # Create left layer of split map
right_layer = geemap.ee_tile_layer(anomalies_day2, anomaly_vis, 'DOF+1 inundation anomalies') # Create right layer of split map
Map.split_map(left_layer, right_layer, left_label='DAY1 Anomalies', right_label='DAY2 Anomalies') # Add left and right layer to the map
Map.add_colorbar(vis_params=anomaly_vis, label="Fractional Inundation Anomaly", orientation="vertical", position="bottomright") # Add colorbar
Map.center_object(roi, 10) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[-5.500068065277028, 105.50000000000024], controls=(ZoomControl(options=['position', 'zoom_in_text'…

#### Split map of CYGNSS data: two months over Mekong

Filter the CYGNSS collection by the months of interest and select the desired band.

In [8]:
april_mean = (
    cygnss_collection
    .map(mask_inundation)
    .filter(ee.Filter.calendarRange(2019, 2025, 'year'))
    .filter(ee.Filter.calendarRange(4, 4, 'month'))
    .mean()
)

october_mean = (
    cygnss_collection
    .map(mask_inundation)
    .filter(ee.Filter.calendarRange(2019, 2025, 'year'))
    .filter(ee.Filter.calendarRange(10, 10, 'month'))
    .mean()
)

map_april = april_mean.select('inundation_3day')
map_october = october_mean.select('inundation_3day')

Set the visualization parameters (colors and value ranges) for the plotted band. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. The code below converts Matplotlib colormaps to HEX color codes. Visualization parameters are defined for inundation observation data (nominal range: 0 to 1).

In [9]:
cmap_inundation = mpl.colormaps["gist_earth_r"].resampled(256) # palette (gist_earth) from matplotlib (https://matplotlib.org/stable/users/explain/colors/colormaps.html)
palette_inundation_hex = [mpl.colors.rgb2hex(cmap_inundation(i)) for i in range(256)]

inundation_vis = {
    'min': 0,
    'max': 1,
    'palette': palette_inundation_hex
}

Plot split map with CYGNSS inundation anomalies from two different months, over the region of interest

In [10]:
# Region of interest – Indochina Peninsula
roi = ee.Geometry.Rectangle([
    94, 7,   # west, south
    110, 19     # east, north
])

Map = geemap.Map() # generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
left_layer = geemap.ee_tile_layer(map_april, inundation_vis, "Fractional Inundation April") # Create left layer of split map
right_layer = geemap.ee_tile_layer(map_october, inundation_vis, "Fractional Inundation October") # Create right layer of split map
Map.split_map(left_layer, right_layer, left_label='April', right_label='October') # Add left and right layer to the map
Map.add_colorbar(inundation_vis, label="Fractional Inundation", orientation="vertical", position="bottomleft") # Add colorbar to the left 
Map.add_colorbar(inundation_vis, label="Fractional Inundation", orientation="vertical", position="bottomright") # Add colorbar to the left 
Map.center_object(roi, 6) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[13.072083244961402, 102], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_titl…